# Notebook Phase 1
Preparation de la base SQLight

## Création des tables SQLite

In [ ]:
import sqlite3

# Connexion à la base de données
conn = sqlite3.connect("..\db\retailsense.db")
cursor = conn.cursor()

# Activer les clés étrangères
cursor.execute("PRAGMA foreign_keys = ON;")

# ==========================
# Table geolocation
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_id INTEGER PRIMARY KEY AUTOINCREMENT,
    geolocation_zip_code_prefix TEXT,
    geolocation_lat REAL,
    geolocation_lng REAL,
    geolocation_city TEXT,
    geolocation_state TEXT
);
""")

# ==========================
# Table product_category_name_translation
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS product_category_name_translation (
    product_category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_category_name TEXT,
    product_category_name_english TEXT
);
""")

# ==========================
# Table customers
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT,
    customer_zip_code_prefix TEXT,
    customer_city TEXT,
    customer_state TEXT
);
""")

# ==========================
# Table sellers
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix TEXT,
    seller_city TEXT,
    seller_state TEXT
);
""")

# ==========================
# Table products
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_lenght INTEGER,
    product_description_lenght INTEGER,
    product_photos_qty INTEGER,
    product_weight_g REAL,
    product_length_cm REAL,
    product_height_cm REAL,
    product_width_cm REAL,
    FOREIGN KEY (product_category_name)
        REFERENCES product_category_name_translation(product_category_name)
);
""")

# ==========================
# Table orders
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_status TEXT,
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME,
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);
""")

# ==========================
# Table order_items
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id TEXT,
    order_item_sequence INTEGER,
    product_id TEXT,
    seller_id TEXT,
    shipping_limit_date DATETIME,
    price REAL,
    freight_value REAL,
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),
    FOREIGN KEY (product_id)
        REFERENCES products(product_id),
    FOREIGN KEY (seller_id)
        REFERENCES sellers(seller_id)
);
""")

# ==========================
# Table payments
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id TEXT,
    payment_sequential INTEGER,
    payment_type TEXT,
    payment_installments INTEGER,
    payment_value REAL,
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
);
""")

# ==========================
# Table reviews
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS reviews (
    review_id TEXT PRIMARY KEY,
    order_id TEXT,
    review_score INTEGER,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME,
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
);
""")

# Sauvegarder les changements
conn.commit()

print("Toutes les tables ont été créées avec succès.")

# Juste pour vérifier
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("\nTables in the database:", cursor.fetchall())

# Fermer la connexion
conn.close()

Toutes les tables ont été créées avec succès.

Tables in the database: [('geolocation',), ('sqlite_sequence',), ('product_category_name_translation',), ('customers',), ('sellers',), ('products',), ('orders',), ('order_items',), ('payments',), ('reviews',)]


## Chargement des données dans SQLite

In [ ]:
import pandas as pd
import sqlite3
import time

start = time.time()

conn = sqlite3.connect("..\db\retailsense.db")
conn.execute("PRAGMA foreign_keys = ON")

print("Reading CSV files...")

geo = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
translation = pd.read_csv("../data/raw/product_category_name_translation.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

print(f"CSV files read in {time.time() - start:.2f} seconds")

# ==========================
# CLEANING
# ==========================
print("\nApplying data cleaning...")

# reviews duplicates (review_id n'est PAS unique dans dataset)
reviews = reviews.drop_duplicates()

# NaN reviews
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("No title")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("No comment")

print("Cleaning done.")

# ==========================
# LOAD DATA
# ==========================
print("\nLoading data into SQLite...")

translation.to_sql("product_category_name_translation", conn, if_exists="replace", index=False)
geo.to_sql("geolocation", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
sellers.to_sql("sellers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
payments.to_sql("payments", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)

conn.commit()
conn.close()

print("Import terminé proprement.")

Reading CSV files...
CSV files read in 3.08 seconds

Applying data cleaning...
Cleaning done.

Loading data into SQLite...
Import terminé proprement.


## Agrégations SQL pour l'analyse des données

In [ ]:
import sqlite3
import pandas as pd

DB_FILE = '..\db\retailsense.db'

def run_query(db_file, query):
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        df = pd.read_sql_query(query, conn)
        return df
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        return None
    finally:
        if conn:
            conn.close()

print("Executing SQL aggregation queries:")

# 1. Chiffre d'affaires par client
print("\n--- Revenue by Customer ---")
query_revenue_by_customer = """
SELECT
    c.customer_unique_id,
    SUM(oi.price + oi.freight_value) AS total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
ORDER BY total_revenue DESC;
"""
df_revenue_by_customer = run_query(DB_FILE, query_revenue_by_customer)
if df_revenue_by_customer is not None:
    print(df_revenue_by_customer.head())

# 2. Chiffre d'affaires par produit
print("\n--- Revenue by Product ---")
query_revenue_by_product = """
SELECT
    p.product_category_name,
    SUM(oi.price + oi.freight_value) AS total_revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_category_name
ORDER BY total_revenue DESC;
"""
df_revenue_by_product = run_query(DB_FILE, query_revenue_by_product)
if df_revenue_by_product is not None:
    print(df_revenue_by_product.head())

# 3. Chiffre d'affaires par mois
print("\n--- Revenue by Month ---")
query_revenue_by_month = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS sale_month,
    SUM(oi.price + oi.freight_value) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY sale_month
ORDER BY sale_month;
"""
df_revenue_by_month = run_query(DB_FILE, query_revenue_by_month)
if df_revenue_by_month is not None:
    print(df_revenue_by_month.head())

# 4. Nombre de commandes
print("\n--- Number of Orders ---")
query_number_of_orders = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders
FROM orders;
"""
df_number_of_orders = run_query(DB_FILE, query_number_of_orders)
if df_number_of_orders is not None:
    print(df_number_of_orders)

# 5. Dernier achat par client
print("\n--- Last Purchase by Customer ---")
query_last_purchase = """
SELECT
    c.customer_unique_id,
    MAX(o.order_purchase_timestamp) AS last_purchase_date
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY last_purchase_date DESC;
"""
df_last_purchase = run_query(DB_FILE, query_last_purchase)
if df_last_purchase is not None:
    print(df_last_purchase.head())

Executing SQL aggregation queries:

--- Revenue by Customer ---
                 customer_unique_id  total_revenue
0  0a0a92112bd4c708ca5fde585afaa872       13664.08
1  da122df9eeddfedc1dc1f5349a1a690c        7571.63
2  763c8b1c9c68a0229c42c9fc6f662b93        7274.88
3  dc4802a71eae9be1dd28f5d788ceb526        6929.31
4  459bef486812aa25204be022145caa62        6922.21

--- Revenue by Product ---
    product_category_name  total_revenue
0            beleza_saude     1441248.07
1      relogios_presentes     1305541.61
2         cama_mesa_banho     1241681.72
3           esporte_lazer     1156656.48
4  informatica_acessorios     1059272.40

--- Revenue by Month ---
  sale_month  total_revenue
0    2016-09         354.75
1    2016-10       56808.84
2    2016-12          19.62
3    2017-01      137188.49
4    2017-02      286280.62

--- Number of Orders ---
   total_orders
0         99441

--- Last Purchase by Customer ---
                 customer_unique_id   last_purchase_date
0  87ab9fec9

## Vérification de l'intégrité des données et Indexation

In [ ]:
import sqlite3
import pandas as pd

DB_FILE = '..\db\retailsense.db'

def run_query(db_file, query):
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        df = pd.read_sql_query(query, conn)
        return df
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        return None
    finally:
        if conn:
            conn.close()

print("Performing data integrity checks and indexing:")

# 1. Vérifier l'intégrité (clés, doublons, valeurs nulles)

# Doublons PK (orders.order_id)
print("\n--- Checking for Primary Key Duplicates (orders.order_id) ---")
query_pk_duplicates = """
SELECT order_id, COUNT(*) as count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;
"""
df_pk_duplicates = run_query(DB_FILE, query_pk_duplicates)
if df_pk_duplicates.empty:
    print("No primary key duplicates found for 'orders.order_id'.")
else:
    print("Warning: Primary key duplicates found for 'orders.order_id':")
    print(df_pk_duplicates)

# Nulls colonnes critiques (customers)
print("\n--- Checking for Null Values in Critical Columns (customers) ---")
critical_customer_cols = ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix']
for col in critical_customer_cols:
    query_null_check = f"""
    SELECT COUNT(*) FROM customers WHERE {col} IS NULL;
    """
    null_count = run_query(DB_FILE, query_null_check).iloc[0,0]
    if null_count > 0:
        print(f"Warning: Column '{col}' in 'customers' has {null_count} NULL values.")
    else:
        print(f"Column '{col}' in 'customers' has no NULL values.")

# FK orphelines (orders.customer_id)
# SQLite ne force pas les clés étrangères par défaut.
# On peut chercher les enregistrements "orphelins".
print("\n--- Checking for Orphaned Foreign Keys (orders.customer_id) ---")
query_fk_orphaned = """
SELECT COUNT(o.customer_id)
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
"""
orphaned_count = run_query(DB_FILE, query_fk_orphaned).iloc[0,0]
if orphaned_count > 0:
    print(f"Found {orphaned_count} orders with a customer_id that does not exist in the 'customers' table. This indicates an issue.")
else:
    print("No orphaned records found for 'orders.customer_id'.")

# 2. Indexer les colonnes fréquemment filtrées

# Index colonnes souvent utilisées
print("\n--- Creating Indexes for frequently filtered columns ---")

def create_index(db_file, table_name, column_name):
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        index_name = f"idx_{table_name}_{column_name}"
        # Vérifie si index existe
        cursor.execute(f"PRAGMA index_list('{table_name}');")
        existing_indexes = [idx[1] for idx in cursor.fetchall()]

        if index_name not in existing_indexes:
            cursor.execute(f"CREATE INDEX {index_name} ON {table_name} ({column_name});")
            conn.commit()
            print(f"Index '{index_name}' created on {table_name}({column_name}).")
        else:
            print(f"Index '{index_name}' already exists. Skipping creation.")
    except sqlite3.Error as e:
        print(f"Error creating index {index_name}: {e}")
    finally:
        if conn:
            conn.close()

create_index(DB_FILE, 'customers', 'customer_unique_id')
create_index(DB_FILE, 'orders', 'order_purchase_timestamp')
create_index(DB_FILE, 'products', 'product_category_name')
create_index(DB_FILE, 'geolocation', 'geolocation_zip_code_prefix')
create_index(DB_FILE, 'sellers', 'seller_zip_code_prefix')

Performing data integrity checks and indexing:

--- Checking for Primary Key Duplicates (orders.order_id) ---
No primary key duplicates found for 'orders.order_id'.

--- Checking for Null Values in Critical Columns (customers) ---
Column 'customer_id' in 'customers' has no NULL values.
Column 'customer_unique_id' in 'customers' has no NULL values.
Column 'customer_zip_code_prefix' in 'customers' has no NULL values.

--- Checking for Orphaned Foreign Keys (orders.customer_id) ---
No orphaned records found for 'orders.customer_id'.

--- Creating Indexes for frequently filtered columns ---
Index 'idx_customers_customer_unique_id' created on customers(customer_unique_id).
Index 'idx_orders_order_purchase_timestamp' created on orders(order_purchase_timestamp).
Index 'idx_products_product_category_name' created on products(product_category_name).
Index 'idx_geolocation_geolocation_zip_code_prefix' created on geolocation(geolocation_zip_code_prefix).
Index 'idx_sellers_seller_zip_code_prefix'

## Suppression des doublons et vérification finale

In [ ]:
import sqlite3
import pandas as pd

DB_FILE = '..\db\retailsense.db'

def run_query(db_file, query):
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        df = pd.read_sql_query(query, conn)
        return df
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        return None
    finally:
        if conn:
            conn.close()

print("Addressing duplicate 'order_id' entries in 'orders' table...")

# On supprime les doublons d'order_id, en gardant le premier
conn = None
try:
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    # Table temp. order_id uniques
    cursor.execute("""
    CREATE TEMPORARY TABLE temp_orders AS
    SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_purchase_timestamp) as rn
    FROM orders;
    """
    )

    # Vide table originale
    cursor.execute("DELETE FROM orders;")

    # Réinsère uniques
    cursor.execute("""
    INSERT INTO orders SELECT
        order_id,
        customer_id,
        order_status,
        order_purchase_timestamp,
        order_approved_at,
        order_delivered_carrier_date,
        order_delivered_customer_date,
        order_estimated_delivery_date
    FROM temp_orders WHERE rn = 1;
    """
    )

    conn.commit()
    print("Duplicate 'order_id' entries removed from 'orders' table.")

except sqlite3.Error as e:
    print(f"SQLite error during duplicate removal: {e}")
finally:
    if conn:
        conn.close()

# Revérifie sans doublons
print("\nVerifying 'orders' table after duplicate removal...")
query_pk_duplicates_after_fix = """
SELECT order_id, COUNT(*) as count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;
"""
df_pk_duplicates_after_fix = run_query(DB_FILE, query_pk_duplicates_after_fix)
if df_pk_duplicates_after_fix.empty:
    print("Successfully removed all duplicate primary keys from 'orders.order_id'.")
else:
    print("Warning: Primary key duplicates still found for 'orders.order_id' after attempted fix:")
    print(df_pk_duplicates_after_fix)

# Vérifie nb total commandes
query_count_after_dedup = "SELECT COUNT(*) FROM orders;"
df_count_after_dedup = run_query(DB_FILE, query_count_after_dedup)
print(f"Total orders after deduplication: {df_count_after_dedup.iloc[0,0]}")

Addressing duplicate 'order_id' entries in 'orders' table...
Duplicate 'order_id' entries removed from 'orders' table.

Verifying 'orders' table after duplicate removal...
Successfully removed all duplicate primary keys from 'orders.order_id'.
Total orders after deduplication: 99441


& .\venv\Scripts\Activate.ps1; New-Item -ItemType Directory -Path .\notebooks\runs -Force; papermill .\notebooks\Phase_1_create_db_SQLight.ipynb .\notebooks\runs\Phase_1_create_db_SQLight_run.ipynb -k retailsense-venv --log-output
